# 🍎 Fresh & Rotten Dataset — Augmentation + Train/Val Split
> **Augments each class to 1200 images → 1000 train / 200 val**  
> No black borders. Only clean, CV-proven augmentations.

---

In [ ]:
# ✅ Install required libraries (run once)
# Pillow and numpy are usually pre-installed in Jupyter
import importlib, subprocess, sys

required = ['Pillow', 'numpy', 'opencv-python']
for pkg in required:
    try:
        importlib.import_module(pkg.lower().replace('-','').replace('opencv','cv2').replace('pythoncv2','cv2').replace('opencvpython','cv2'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All libraries ready!')

In [ ]:
import os, random, shutil
from pathlib import Path
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('✅ Imports done!')

## ⚙️ Configuration
**Only change this cell** — set your dataset path and counts.

In [ ]:
# ──────────────────────────────────────────────
# CHANGE THESE VALUES as per your setup
# ──────────────────────────────────────────────

DATASET_DIR  = './dataset'   # 📂 Path to your dataset folder
TARGET_TOTAL = 1200          # Total images per class after augmentation
TRAIN_COUNT  = 1000          # Images in train split
VAL_COUNT    = 200           # Images in val split
IMG_SIZE     = (224, 224)    # Resize all images to this (H, W)
SEED         = 42            # Reproducibility seed

SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

assert TRAIN_COUNT + VAL_COUNT == TARGET_TOTAL, '❌ Train + Val must equal TARGET_TOTAL!'

random.seed(SEED)
np.random.seed(SEED)

print(f'✅ Config set: {TARGET_TOTAL} total → {TRAIN_COUNT} train + {VAL_COUNT} val')
print(f'📂 Dataset path: {Path(DATASET_DIR).resolve()}')

## 🔧 Augmentation Functions

Each transform has a **probability** (e.g., 50%) of being applied per image.  
Multiple transforms can stack, creating diverse variations.

| Augmentation | Probability | Why it helps |
|---|---|---|
| Horizontal Flip | 50% | Fruits appear in any orientation |
| Vertical Flip | 30% | Adds diversity without harm |
| Rotation + crop-resize | 60% | Avoids black corners by cropping |
| Brightness | 60% | Different lighting conditions |
| Contrast | 50% | Camera/sensor differences |
| Saturation (Color) | 50% | Camera color rendering varies |
| Sharpness | 40% | Blur or sharp cameras |
| Gaussian Blur | 30% | Defocus simulation |
| Gaussian Noise | 30% | Sensor noise in low-light |
| Random Crop + Resize | 40% | Teaches texture focus, simulates zoom |


In [ ]:
def augment_image(img: Image.Image) -> Image.Image:
    """
    Applies a random combination of CV augmentations.
    No black borders/padding used anywhere.
    """
    img = img.convert('RGB')

    # 1. Horizontal Flip
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)

    # 2. Vertical Flip
    if random.random() < 0.3:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)

    # 3. Rotation — expand then resize avoids black corners
    if random.random() < 0.6:
        angle = random.uniform(-25, 25)
        img = img.rotate(angle, resample=Image.BILINEAR, expand=True)
        img = img.resize(IMG_SIZE, Image.BILINEAR)

    # 4. Brightness
    if random.random() < 0.6:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.5))

    # 5. Contrast
    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.5))

    # 6. Saturation (Color)
    if random.random() < 0.5:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.5, 1.8))

    # 7. Sharpness
    if random.random() < 0.4:
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.5, 2.5))

    # 8. Gaussian Blur
    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))

    # 9. Gaussian Noise
    if random.random() < 0.3:
        arr = np.array(img, dtype=np.float32)
        noise = np.random.normal(0, random.uniform(5, 15), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)

    # 10. Random Crop + Resize (simulates zoom/partial view)
    if random.random() < 0.4:
        w, h = img.size
        crop_ratio = random.uniform(0.75, 0.95)
        new_w, new_h = int(w * crop_ratio), int(h * crop_ratio)
        left = random.randint(0, w - new_w)
        top  = random.randint(0, h - new_h)
        img = img.crop((left, top, left + new_w, top + new_h))
        img = img.resize(IMG_SIZE, Image.BILINEAR)

    return img.resize(IMG_SIZE, Image.BILINEAR)


print('✅ augment_image() function defined!')

## 👁️ Preview Augmentations
Run this cell to see what augmentations look like on a sample image before processing.

In [ ]:
def preview_augmentations(dataset_dir, n_aug=8):
    """Shows original + n augmented versions of a random image."""
    dataset_path = Path(dataset_dir)
    all_imgs = list(dataset_path.rglob('*'))
    all_imgs = [p for p in all_imgs if p.suffix.lower() in SUPPORTED_EXTS
                and 'train' not in str(p) and 'val' not in str(p)]

    if not all_imgs:
        print('❌ No images found for preview!')
        return

    sample = random.choice(all_imgs)
    original = Image.open(sample).convert('RGB').resize(IMG_SIZE)

    fig, axes = plt.subplots(2, (n_aug + 2) // 2, figsize=(16, 6))
    axes = axes.flatten()

    axes[0].imshow(original)
    axes[0].set_title('ORIGINAL', fontweight='bold', color='green')
    axes[0].axis('off')

    for i in range(1, n_aug + 1):
        aug = augment_image(original.copy())
        axes[i].imshow(aug)
        axes[i].set_title(f'Augmented {i}')
        axes[i].axis('off')

    for j in range(n_aug + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Augmentation Preview\nSource: {sample.name}', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f'✅ Preview done! Source image: {sample}')

# Run preview
preview_augmentations(DATASET_DIR)

## 📊 Step 1 — Scan Dataset
Let's first count what we have before augmenting.

In [ ]:
def scan_dataset(dataset_dir):
    """Scans and reports class counts."""
    dataset_path = Path(dataset_dir).resolve()

    if not dataset_path.exists():
        raise FileNotFoundError(f'❌ Dataset folder not found: {dataset_path}')

    skip_dirs = {'train', 'val'}
    class_dirs = [d for d in dataset_path.iterdir()
                  if d.is_dir() and d.name.lower() not in skip_dirs]

    if not class_dirs:
        raise ValueError(f'❌ No class subfolders found in {dataset_path}')

    print(f'📂 Dataset path : {dataset_path}')
    print(f'🗂️  Classes found : {[d.name for d in class_dirs]}\n')

    total = 0
    for cls in class_dirs:
        imgs = [p for p in cls.iterdir() if p.suffix.lower() in SUPPORTED_EXTS]
        needed = max(0, TARGET_TOTAL - len(imgs))
        print(f'  📁 {cls.name:15s} → {len(imgs):>5} images  |  Need to generate: {needed}')
        total += len(imgs)

    print(f'\n  Total original images: {total}')
    print(f'  Target after augment : {TARGET_TOTAL} per class')
    return class_dirs

class_dirs = scan_dataset(DATASET_DIR)

## 🚀 Step 2 — Augment + Split
This cell does the heavy lifting. It will:
1. Load all original images
2. Generate augmented images to reach 1200
3. Shuffle and split 1000 → train, 200 → val
4. Save everything to the output folders


In [ ]:
def process_class(class_name, src_folder, train_out, val_out):
    print(f'\n{'='*50}')
    print(f'Processing: {class_name}')
    print(f'{'='*50}')

    # Collect original images
    images = [p for p in src_folder.iterdir() if p.suffix.lower() in SUPPORTED_EXTS]
    print(f'  Found {len(images)} original images')

    if len(images) == 0:
        print(f'  [SKIP] No images found. Skipping.')
        return

    # Load originals as PIL
    augmented_pool = []
    for img_path in images:
        try:
            img = Image.open(img_path).convert('RGB').resize(IMG_SIZE, Image.BILINEAR)
            augmented_pool.append((img, img_path.name))
        except Exception as e:
            print(f'  [WARN] Skipping {img_path.name}: {e}')

    print(f'  Loaded {len(augmented_pool)} images successfully')

    # Augment or sample down to TARGET_TOTAL
    if len(augmented_pool) >= TARGET_TOTAL:
        print(f'  Already {len(augmented_pool)} images — sampling down to {TARGET_TOTAL}')
        augmented_pool = random.sample(augmented_pool, TARGET_TOTAL)
    else:
        needed = TARGET_TOTAL - len(augmented_pool)
        print(f'  Generating {needed} augmented images...')
        src_pils = [item[0] for item in augmented_pool]
        count = 0
        while len(augmented_pool) < TARGET_TOTAL:
            base = random.choice(src_pils)
            aug  = augment_image(base.copy())
            augmented_pool.append((aug, f'aug_{count:05d}.jpg'))
            count += 1
        print(f'  ✅ Pool ready: {len(augmented_pool)} images')

    # Shuffle and split
    random.shuffle(augmented_pool)
    train_pool = augmented_pool[:TRAIN_COUNT]
    val_pool   = augmented_pool[TRAIN_COUNT:TRAIN_COUNT + VAL_COUNT]

    # Create output dirs
    train_dir = train_out / class_name
    val_dir   = val_out   / class_name
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)

    # Save images
    def save_pool(pool, out_dir, split_name):
        saved = 0
        for idx, (img, name) in enumerate(pool):
            save_path = out_dir / f'{Path(name).stem}_{idx:04d}.jpg'
            try:
                img.save(save_path, 'JPEG', quality=92)
                saved += 1
            except Exception as e:
                print(f'  [WARN] Save failed for {save_path.name}: {e}')
        print(f'  💾 {split_name}: {saved} images saved → {out_dir}')

    save_pool(train_pool, train_dir, 'Train')
    save_pool(val_pool,   val_dir,   'Val')


# ── Run for all classes ──
dataset_path = Path(DATASET_DIR).resolve()
train_out    = dataset_path / 'train'
val_out      = dataset_path / 'val'

for class_dir in class_dirs:
    process_class(class_dir.name, class_dir, train_out, val_out)

print('\n🎉 ALL DONE!')

## ✅ Step 3 — Verify Output
Let's confirm the final counts in train and val folders.

In [ ]:
def verify_output(dataset_dir):
    dataset_path = Path(dataset_dir).resolve()
    print('📊 Final Dataset Summary')
    print('=' * 45)

    for split in ['train', 'val']:
        split_path = dataset_path / split
        if not split_path.exists():
            print(f'  {split.upper():6s}: ❌ Not found')
            continue

        print(f'\n  📁 {split.upper()}')
        split_total = 0
        for cls_dir in sorted(split_path.iterdir()):
            if cls_dir.is_dir():
                count = len([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
                print(f'     {cls_dir.name:15s}: {count:>5} images')
                split_total += count
        print(f'     {"TOTAL":15s}: {split_total:>5} images')

verify_output(DATASET_DIR)

## 📈 Step 4 — Visualize Class Distribution

In [ ]:
def plot_distribution(dataset_dir):
    dataset_path = Path(dataset_dir).resolve()
    splits = ['train', 'val']
    data = {}

    for split in splits:
        split_path = dataset_path / split
        if not split_path.exists(): continue
        data[split] = {}
        for cls_dir in sorted(split_path.iterdir()):
            if cls_dir.is_dir():
                count = len([p for p in cls_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
                data[split][cls_dir.name] = count

    if not data:
        print('No output data found to plot.')
        return

    classes = sorted(set(cls for s in data.values() for cls in s))
    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#4CAF50', '#2196F3']

    for i, split in enumerate(splits):
        counts = [data.get(split, {}).get(c, 0) for c in classes]
        bars = ax.bar(x + i * width, counts, width, label=split.upper(), color=colors[i], alpha=0.85)
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                    str(count), ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_xlabel('Class', fontsize=12)
    ax.set_ylabel('Image Count', fontsize=12)
    ax.set_title('Train vs Val Split per Class', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(classes, fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_distribution(DATASET_DIR)